# Lesson 9: Running a Pretrained Language Model

In the earlier Weird AI lessons, you built many of the core pieces of a language model from scratch. In this lesson, you will use a pretrained model so you can focus on the complete text-generation pipeline.

The pipeline is:

```text
prompt text -> tokenizer -> token IDs -> pretrained model -> generated token IDs -> generated text
```

By the end of this notebook, you should be able to explain how a prompt becomes token IDs, how a pretrained model generates more tokens, and how those generated tokens become readable text again.

## Section 1: Setup

Run this notebook from the project root folder. If imports fail, make sure your virtual environment is active and the project is installed in editable mode:

```bash
python -m pip install -e .
python -m pip install -r requirements.txt
```

This notebook uses Hugging Face `transformers`. If it is not installed yet, install it with:

```bash
python -m pip install transformers
```

In [1]:
from pathlib import Path
import torch

from weird_ai.pretrained import (
    DEFAULT_MODEL_NAME,
    get_device,
    load_tokenizer,
    load_model,
    tokenize_prompt,
    decode_generated_tokens,
    generate_text,
    generate_with_reasoning,
)

from weird_ai.reasoning import build_reasoning_prompt

## Section 2: Check Your Hardware

A GPU is helpful for larger models, but this lesson should work on CPU with a very small demonstration model. The starter code should choose CUDA when available, Apple Silicon MPS when available, and CPU otherwise.

In [2]:
device = get_device()
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

Using device: cuda
PyTorch version: 2.10.0+cu128


## Section 3: Load a Tokenizer

A tokenizer converts human-readable text into token IDs. The model does not directly understand strings such as `Write a parody about networking`. It receives integers.

For this notebook, the default model is intentionally tiny so the first download is fast.

In [3]:
tokenizer = load_tokenizer(DEFAULT_MODEL_NAME)
print(f"Loaded tokenizer for: {DEFAULT_MODEL_NAME}")
print(f"Pad token: {tokenizer.pad_token!r}")
print(f"EOS token: {tokenizer.eos_token!r}")

Loaded tokenizer for: sshleifer/tiny-gpt2
Pad token: '<|endoftext|>'
EOS token: '<|endoftext|>'


## Section 4: Tokenization Round Trip

A round trip means:

```text
text -> token IDs -> text
```

This helps verify that the tokenizer can encode and decode text.

In [ ]:
prompt = "Write a parody chorus about a student debugging Python at midnight."

# Encode the prompt using tokenizer.encode().
token_ids = tokenizer.encode(prompt)

# Decode the token IDs back into text using tokenizer.decode().
decoded_text = tokenizer.decode(token_ids)

print("Original text:")
print(prompt)
print("\nToken IDs:")
print(token_ids)
print("\nDecoded Text")
print(decoded_text)

Original text:
Write a parody chorus about a student debugging Python at midnight.

Token IDs:
[16594, 257, 29695, 26441, 546, 257, 3710, 28769, 11361, 379, 15896, 13]

Decoded Text
Write a parody chorus about a student debugging Python at midnight.


### Reflection Check

Answer in your assignment document:

Why does the model need token IDs instead of regular text?
- The model works with numbers, it cannot read text directly.


## Section 5: Inspect Individual Tokens

Many modern tokenizers use subword tokens. A word may be one token, or it may be split into multiple pieces.

In [6]:
for token_id in token_ids:
    piece = tokenizer.decode([token_id])
    print(f"{token_id:>6} -> {piece!r}")

 16594 -> 'Write'
   257 -> ' a'
 29695 -> ' parody'
 26441 -> ' chorus'
   546 -> ' about'
   257 -> ' a'
  3710 -> ' student'
 28769 -> ' debugging'
 11361 -> ' Python'
   379 -> ' at'
 15896 -> ' midnight'
    13 -> '.'


## Section 6: Load a Pretrained Model

A pretrained model already has learned statistical patterns from a large training corpus. In this lesson, you are not training the model. You are using it for inference.

This can take a moment the first time because the model weights need to download.

In [8]:
model = load_model(DEFAULT_MODEL_NAME, device=device)
print(f"Loaded model for: {DEFAULT_MODEL_NAME}")

Loading weights:   0%|          | 0/29 [00:00<?, ?it/s]

[transformers] GPT2LMHeadModel LOAD REPORT from: sshleifer/tiny-gpt2
Key                                   | Status     |  | 
--------------------------------------+------------+--+-
transformer.h.{0, 1}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded model for: sshleifer/tiny-gpt2


## Section 7: Generate Text

Now you will complete the full pipeline:

```text
prompt -> tokenizer -> model.generate() -> decode
```

The `generate_text()` function in `pretrained.py` should handle those steps.

In [9]:
prompt = "Write a short parody chorus about forgetting to commit code before a demo:"

output = generate_text(
    prompt,
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=50,
    temperature=0.8,
    top_k=50,
    device=device,
)

print(output)

Write a short parody chorus about forgetting to commit code before a demo:reement directly reviewing confir trilogy vendors intermittent ESVJD Participation subst conservationScene circumcised Daniel hauled heir scalpatisfJD confir subst Brew intermittenthibit Hancock Participationikenimura heir conservation credibility trilogy vendors Motorola circumcisedRocketoother004press stairs MoneySher credibilityreement Amph Danielmediately Brew Jr


## Section 8: Compare Temperature Settings

Temperature changes how random generation is.

Lower temperature usually produces safer, more predictable text. Higher temperature usually produces more varied, surprising, and sometimes less coherent text.

In [10]:
prompt = "Write one funny line about a Wi-Fi router with stage fright:"

temperatures = [0.3, 0.8, 1.2]

for temp in temperatures:
    print("=" * 70)
    print(f"Temperature: {temp}")
    print("=" * 70)
    print(generate_text(
        prompt,
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=40,
        temperature=temp,
        top_k=50,
        device=device,
    ))
    print()

Temperature: 0.3
Write one funny line about a Wi-Fi router with stage fright:press Habitditreement circumcised ONE dispatch ONE pawn reborn reviewing pawn stairsoother TAmediately Amph circumcised Participation TAoother dispatchreement hauledreement Hancock Amph Habitmediately stairs Hancockiken scalpohoJD Jr vendors TAreementoother

Temperature: 0.8
Write one funny line about a Wi-Fi router with stage fright:ditmediately conservation Habit Brew ESV Jr antibiotic hauled pawn Participationhibit credibility004 subst Amph conservation substohoScene Habit confir trilogy ONE directly Motorola confir scalpRocket Habit scalpmediately ESV ONE Observ Habitdit reviewingditmediately

Temperature: 1.2
Write one funny line about a Wi-Fi router with stage fright: scalpRocket dispatch Motorola stairs dispatch stairs004reement autonomy antibiotic trilogy Motorola reborn stairs circumcised stairs scalpRocket stairs Prob trilogy conservation intermittentreement intermittent Brew pawn Brew rebornSceneati

## Section 9: Compare Top-k Settings

Top-k sampling limits the model to choosing from only the `k` most likely next tokens.

Smaller values restrict the model more. Larger values allow more variety.

In [11]:
prompt = "Write one strange parody lyric about a database that sings backup vocals:"

top_k_values = [10, 50, 100]

for k in top_k_values:
    print("=" * 70)
    print(f"Top-k: {k}")
    print("=" * 70)
    print(generate_text(
        prompt,
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=40,
        temperature=0.8,
        top_k=k,
        device=device,
    ))
    print()

Top-k: 10
Write one strange parody lyric about a database that sings backup vocals: Jr hauled vendors intermittent stairs Jr stairs stairsdit intermittent Jr Jr hauled Habitditdit Jr stairs Brew Jr stairs stairsditRocketRocketRocket intermittent Rh Brew Rh Brew vendors Habit stairsRocket Rhdit Rh RhRocket

Top-k: 50
Write one strange parody lyric about a database that sings backup vocals: Participation substimura Motorola ParticipationditJD TA conservation JrScene subst circumcised antibiotic Rh004 circumcised Amphmediately Habitatisf vendors Rh Hancock Observdit Brew directly conservation reborn Habit ESV MotorolaootherSher TA subst Rh Hancock ONE

Top-k: 100
Write one strange parody lyric about a database that sings backup vocals: ster proficiency intermittent vendors Prob nonprofitsenna ster Process heir sexualityimura TAJDico Process proficiency vendorsennaoothermediatelyGazailled reborn antibiotic Changedrail landowners unknown heir Rh possibility Daniel intermittent antibiotic a

## Section 10: Generate with a Reasoning Prompt

In Lesson 8, you built prompt helpers that ask Weird AI to think step-by-step. Now you can combine that reasoning prompt with a pretrained model.

In [12]:
user_prompt = "Create a parody song idea about students learning recursion."

reasoning_prompt = build_reasoning_prompt(user_prompt)
print(reasoning_prompt)

You are a Weird AI.
Think step-by-step.
Question: Create a parody song idea about students learning recursion.
Final Answer:



In [13]:
reasoning_output = generate_with_reasoning(
    user_prompt,
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=100,
    temperature=0.8,
    top_k=50,
    device=device,
)

print(reasoning_output)

You are a Weird AI.
Think step-by-step.
Question: Create a parody song idea about students learning recursion.
Final Answer:
 Singaporeacious representations Tre skillet grandchildren skillet perhaps Bend Tre grandchildren bravery Dreams incarcer boils skillet omegaGyOutside mutual omega courtyardived equate rented soy Tre bravery 236Pros linedGy Bend Wheels prayingshows Dreams653 factors skillet boilsMini factors Television rubbingivedshowsozygMini soy Pocket rented Wheelsobl courtyard representations courtyard bravery omegaMini653 praying workshopsSexualobl Medic grandchildren653 workshops Dreams clearer grandchildrenGy boils clearer rented 236 bravery653 Televisionived courtyardoblshows� Singapore courtyard grandchildren PocketMost equate representations Medic deflect praying Singapore Television courtyard Singapore courtyard


## Section 11: Weird AI Experiment

Choose one of your own parody topics. Generate at least two outputs using different settings. Save the prompts, settings, and outputs in your assignment document.

In [15]:
my_topic = "a student who is panicking about finding a job a month before graduation"

settings = [
    {"temperature": 0.5, "top_k": 25},
    {"temperature": 1.0, "top_k": 100},
]

for setting in settings:
    print("=" * 70)
    print(setting)
    print("=" * 70)
    print(generate_text(
        f"Write a short parody verse about {my_topic}:",
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=80,
        temperature=setting["temperature"],
        top_k=setting["top_k"],
        device=device,
    ))
    print()

{'temperature': 0.5, 'top_k': 25}
Write a short parody verse about a student who is panicking about finding a job a month before graduation:ohodit Habitoho intermittent stairs antibiotic Habit stairs autonomy Jr autonomy confir stairsRocketdit subst subst hauledpresshibitJD autonomyhibit reviewing Prob reviewing intermittent intermittentoho subst trilogy confir Habit reviewing confir credibility confir Brew Brew Brew Daniel vendors intermittent ProbRocket HabitRocket Rh Daniel ESV autonomy stairsdit trilogyohoJDRocketditpress hauled trilogy intermittentpresshibit conservationRocket Prob vendors substdit vendorsRocket vendors stairs stairs Rh Daniel Rh trilogy

{'temperature': 1.0, 'top_k': 100}
Write a short parody verse about a student who is panicking about finding a job a month before graduation: Participation dissolutionoho dispatch TA necessarily compromisedormaniabilityanity TMZ Rh Hancock004 trilogystanbulimurarawdownload directlyilled circumcised TMZteriorenna="rail dispatchiab

## Section 12: Lesson Summary

In this notebook, you practiced:

- Loading a pretrained tokenizer
- Encoding text into token IDs
- Decoding token IDs back into text
- Loading a pretrained causal language model
- Generating text from a prompt
- Comparing temperature and top-k settings
- Reusing a Lesson 8 reasoning prompt with a pretrained model

Before submitting, run the unit tests:

```bash
python -m pytest tests/test_pretrained.py
```